# CSC 349 HW4 — Analysis

This separate notebook answers the five analysis questions using the results saved by Task 8 of `heart_disease_classification.ipynb`. Run that notebook first, then place its `heart_disease_results.json` beside this notebook or upload it when prompted. The following code displays the actual saved metrics before the answers. It does not retrain models or invent missing results.

**Execution status:** Analysis executed against the actual local verification results. Colab execution remains pending.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
result_path = Path('heart_disease_results.json')
if not result_path.exists():
    from google.colab import files
    uploaded = files.upload()
    if result_path.name not in uploaded:
        raise FileNotFoundError('Upload heart_disease_results.json from Task 8.')
r = json.loads(result_path.read_text())
display(pd.DataFrame(r['metrics']).T.drop(columns='confusion_matrix'))
print('Best parameters:', r['best_params'])
print('Best cross-validation AUC:', r['best_cv_auc'])
display(pd.Series(r['grouped_importance'], name='Grouped importance').to_frame())

Best parameters: {'model__C': 100, 'model__gamma': 0.001}
Best cross-validation AUC: 0.9026992331340157


,accuracy,precision,recall,f1,roc_auc
Baseline SVC,0.885246,0.83871,0.928571,0.881356,0.964286
Decision Tree,0.737705,0.676471,0.821429,0.741935,0.744048
Tuned SVC,0.852459,0.806452,0.892857,0.847458,0.948052


,Grouped importance
thal,0.292921
cp,0.144734
ca,0.111887
chol,0.110055
oldpeak,0.107935
age,0.096128
sex,0.033909
trestbps,0.027639
thalach,0.018673
fbs,0.016444


## 1. Feature types and the importance of scaling

Heart Disease combines continuous physiological measurements with coded diagnostic categories. Age, cholesterol, resting blood pressure, maximum heart rate, and ST depression represent quantities, while chest pain type, thal, and restecg represent distinct diagnostic findings. Codes such as thal=3, 6, or 7 are labels rather than equally spaced amounts. Even though slope and ca have an order, this assignment encodes them as categories instead of assuming a linear effect.

Compared with the earlier housing dataset, which mixed room and area measurements with yes/no amenities and furnishing categories, the clinical dataset uses more integer-coded diagnostic fields. Like the Telco churn exercise, the storage type alone can be misleading: Telco's TotalCharges contained blank strings, while Cleveland contains question marks. A column that looks numeric still needs an inspection of its meaning and values before modeling.

RBF SVC calculates similarities using squared distances. Without scaling, a feature measured in hundreds can dominate one measured in small fractions because of its units. StandardScaler learns each numerical column's mean and standard deviation from the training fold, then centers and scales it. One-hot encoding represents the categories without assigning numeric distances to their original codes.

A Decision Tree instead asks threshold questions one feature at a time. A positive linear rescaling preserves the ordering of values and therefore the possible partitions, so trees generally do not need numerical scaling. Both models use the same preprocessing here for a controlled comparison, but scaling is especially important to the SVC. Every learned preprocessing step stays inside the pipeline to avoid leakage.

## 2. Hidden missing values and imputation choices

A question mark may be stored as a string rather than a true null. That can cause `isna()` to undercount missingness, force a numerical column into an object type, or fail during fitting. Treating the question mark as a real category would also misrepresent the absence of a clinical measurement. The notebook preserves the raw tokens, explicitly counts them, converts recognized missing markers to NaN, and rejects unexpected nonnumeric tokens.

Median imputation is selected for numerical features because an extreme measurement has less effect on a median than on a mean. It is a reasonable default when some vitals are skewed. It does not mean every numerical variable actually has missing values in this file; the audit below states which ones do. Most-frequent imputation keeps diagnostic fields within their observed category codes, while averaging codes could create a meaningless category. Both methods preserve the small dataset's rows.

Imputers are fitted on training data only, and GridSearchCV refits them separately inside each fold. This avoids learning medians or modes from the test patients. Imputation still cannot recover the true missing observation, and mode imputation can favor the majority category. Missingness may itself relate to testing practices or patient characteristics, so these simple strategies have limitations.

In [2]:
missing = {k:v for k,v in r['missing_counts'].items() if v}
display(Markdown('**Observed missing counts:** ' + ', '.join(f'`{k}`: {v}' for k,v in missing.items()) + '. All other predictors had zero missing values.'))

**Observed missing counts:** `ca`: 4, `thal`: 2. All other predictors had zero missing values.

## 3. RBF SVC versus Decision Tree boundaries and interpretability

The RBF SVC learns a maximum-margin separation in an implicit higher-dimensional feature space. Its kernel compares patients by similarity, allowing curved decision boundaries in the original feature space. Support vectors near the boundary help determine the fitted separation. The model can combine several measurements in a nonlinear way, but it does not provide a short set of clinical rules or ordinary original-feature coefficients. `probability=True` estimates probabilities through an additional calibration procedure; it does not establish clinical calibration in a new population.

A Decision Tree repeatedly chooses a single-feature split that reduces impurity, producing axis-aligned regions. A patient follows a sequence of threshold or one-hot category questions to reach a leaf. This makes individual paths easier to inspect. The plotted numerical thresholds are in standardized units, so they should not be read directly as blood-pressure or heart-rate cutoffs. The displayed plot truncates the view at three levels while the fitted baseline tree remains fully grown.

A large tree can fit noise and change substantially with a different sample. Its rules are easier to inspect than the SVC's boundary, but interpretability decreases as the tree grows. The following comparison uses the same held-out patients for both models, and separates observed performance from any claim of general superiority.

In [3]:
for name, m in r['metrics'].items():
    tn, fp = m['confusion_matrix'][0]
    fn, tp = m['confusion_matrix'][1]
    display(Markdown(f"**{name}:** accuracy {m['accuracy']:.4f}, disease recall {m['recall']:.4f}, F1 {m['f1']:.4f}, ROC-AUC {m['roc_auc']:.4f}. The confusion matrix contains {tn} true negatives, {fp} false positives, {fn} false negatives, and {tp} true positives."))
display(Markdown(f"The tree's training accuracy was **{r['tree_train_accuracy']:.4f}**. Compare this with its test accuracy to assess overfitting. The test set contains only **{r['n_test']} patients**, so a few predictions can noticeably change the reported scores."))

**Baseline SVC:** accuracy 0.8852, disease recall 0.9286, F1 0.8814, ROC-AUC 0.9643. The confusion matrix contains 28 true negatives, 5 false positives, 2 false negatives, and 26 true positives.

**Decision Tree:** accuracy 0.7377, disease recall 0.8214, F1 0.7419, ROC-AUC 0.7440. The confusion matrix contains 22 true negatives, 11 false positives, 5 false negatives, and 23 true positives.

**Tuned SVC:** accuracy 0.8525, disease recall 0.8929, F1 0.8475, ROC-AUC 0.9481. The confusion matrix contains 27 true negatives, 6 false positives, 3 false negatives, and 25 true positives.

The tree's training accuracy was **1.0000**. Compare this with its test accuracy to assess overfitting. The test set contains only **61 patients**, so a few predictions can noticeably change the reported scores.

## 4. C, gamma, bias, variance, and generalization

C controls the penalty for training errors relative to the margin. A smaller C allows more violations and stronger regularization, often giving a simpler boundary with higher bias and lower variance. A larger C penalizes violations more strongly and may fit the training data more closely, increasing variance when it responds to noise. These are tendencies rather than guarantees for every dataset.

Gamma determines how locally each observation influences the RBF similarity. A small gamma spreads influence across a wider neighborhood, usually producing a smoother boundary. A large gamma makes influence more local and can create a very irregular boundary that overfits. C and gamma interact, so this notebook searches combinations rather than changing each one in isolation.

Five stratified training folds select the combination with the highest mean ROC-AUC. The held-out test set is excluded from that selection, as are its preprocessing statistics. The best cross-validation score can be optimistic because it is the maximum over several candidates. Fold variation and the final test result therefore matter alongside the winning mean score. The finite grid also cannot prove that the globally optimal parameters were found. Test results should not be used to repeatedly redesign the grid.

In [4]:
b = r['metrics']['Baseline SVC']; t = r['metrics']['Tuned SVC']
display(Markdown(f"The selected parameters were **{r['best_params']}**. Mean training-fold validation ROC-AUC was **{r['best_cv_auc']:.4f}**, with fold standard deviation **{r['best_cv_std']:.4f}**. Baseline test ROC-AUC was **{b['roc_auc']:.4f}** and tuned test ROC-AUC was **{t['roc_auc']:.4f}** (change **{t['roc_auc']-b['roc_auc']:+.4f}**). A positive change supports improvement on this holdout; a zero or negative change shows that better selected CV performance did not translate into better ranking on these test patients. Neither outcome guarantees performance in a new clinical population."))

The selected parameters were **{'model__C': 100, 'model__gamma': 0.001}**. Mean training-fold validation ROC-AUC was **0.9027**, with fold standard deviation **0.0197**. Baseline test ROC-AUC was **0.9643** and tuned test ROC-AUC was **0.9481** (change **-0.0162**). A positive change supports improvement on this holdout; a zero or negative change shows that better selected CV performance did not translate into better ranking on these test patients. Neither outcome guarantees performance in a new clinical population.

## 5. Strongest clinical indicators and screening implications

For the Decision Tree, impurity importance measures how much each feature reduces weighted impurity across its splits. Because categorical fields become several one-hot columns, their importances are summed back to the original clinical variable before ranking. The RBF SVC has no ordinary linear coefficient vector for this comparison. The actual tree ranking below determines which indicators this analysis identifies as strongest.

In [5]:
ranked = sorted(r['grouped_importance'].items(), key=lambda item:item[1], reverse=True)
display(Markdown('The five largest grouped importances were: ' + '; '.join(f'**{name}** ({value:.4f})' for name,value in ranked[:5]) + '. These are the strongest indicators for this fitted tree under this importance measure.'))
descriptions = {'cp':'chest pain category','thal':'thallium test result','ca':'number of major vessels colored by fluoroscopy', 'thalach':'maximum exercise heart rate','oldpeak':'exercise-related ST depression','exang':'exercise-induced angina','slope':'peak exercise ST-segment slope','age':'patient age','trestbps':'resting blood pressure','chol':'serum cholesterol','restecg':'resting ECG category','fbs':'fasting blood sugar category','sex':'recorded sex category'}
display(Markdown('In clinical terms, the top three variables are ' + ', '.join(descriptions[k] for k,v in ranked[:3]) + '.'))

The five largest grouped importances were: **thal** (0.2929); **cp** (0.1447); **ca** (0.1119); **chol** (0.1101); **oldpeak** (0.1079). These are the strongest indicators for this fitted tree under this importance measure.

In clinical terms, the top three variables are thallium test result, chest pain category, number of major vessels colored by fluoroscopy.

These associations suggest which recorded findings the tree relied on most when separating disease-positive and disease-negative patients. They do not reveal the direction of every effect: a high importance for maximum heart rate, for example, does not by itself say whether higher or lower rates were associated with disease. Reading individual paths would be necessary to understand that tree's conditional rules.

For a cardiologist, these results can motivate further investigation of how the ranked indicators complement symptoms and established risk assessment. They cannot establish a new preventive screening protocol. Some predictors, such as thal and ca, come from specialized diagnostic tests, so this model is not automatically a tool for screening otherwise untested people. Its small historical Cleveland sample, possible selection bias, single split, and simple missing-value assumptions limit generalization. Impurity importance also favors variables with many possible split points and can distribute importance unpredictably among related predictors.

Before using a model to support screening, it would need external validation, subgroup evaluation, probability calibration assessment, and a clinically justified threshold that considers missed disease and unnecessary follow-up. A variable with low importance in this one tree should not be treated as medically unimportant, and no causal or preventive benefit can be inferred from this experiment alone.

## References

- Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart Disease*. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X
- Scikit-learn. [Support Vector Classification](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html).
- Scikit-learn. [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html).